In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv

# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import your package components
from kg_commit.knowledge.dataloader import CommitDataLoader
from kg_commit.knowledge.parsers import FilteredCommitParser
from kg_commit.knowledge.graph import JITCommitKnowledgeGraph

In [3]:
# =====================================================================
# CONFIGURATION & LOCAL PATHS
# =====================================================================
# Target the direct local repository name and path
PROJECT_KEY = "groovy" 
REPO_PATH = f"{REPOS_DIR}/groovy"

# Neo4j Local Instance Credentials
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password1234"

In [4]:
# =====================================================================
# INITIALIZATION
# =====================================================================
repo_map = {PROJECT_KEY: REPO_PATH}
data_loader = CommitDataLoader(repo_map=repo_map)

# Instantiate the structural graph engine using our default parser schema
parser_instance = FilteredCommitParser()
kg = JITCommitKnowledgeGraph(
    uri=NEO4J_URI,
    auth_user=NEO4J_USER,
    auth_pass=NEO4J_PASSWORD,
    parser=parser_instance
)

In [ ]:
# Wipe historical state for a fresh chronological run
print("Wiping historical graph database state...")
kg.purge_database()

# =====================================================================
# DIRECT REPOSITORY LOG STREAMING (CHRONOLOGICAL)
# =====================================================================
print(f"\nOpening local repository at: {REPO_PATH}")
# Access the underlying cached Repo instance inside the loader
repo = data_loader._get_repo(PROJECT_KEY)

# Stream all commits from the repository, ordered from past to present
print("Extracting commit timeline in ascending chronological order (oldest -> newest)...")
chronological_commits = list(repo.iter_commits(reverse=True))
total_commits = len(chronological_commits)

print(f"Discovered a total of {total_commits} historical commits inside the repo history.\n")
print("=" * 60)

ingested_count = 0
skipped_count = 0

from tqdm import tqdm

# Wipe historical state for a fresh chronological run
print("Wiping historical graph database state...")
kg.purge_database()

# =====================================================================
# DIRECT REPOSITORY LOG STREAMING (CHRONOLOGICAL WITH TQDM)
# =====================================================================
print(f"\nOpening local repository at: {REPO_PATH}")
repo = data_loader._get_repo(PROJECT_KEY)

print("Extracting commit timeline in ascending chronological order (oldest -> newest)...")
chronological_commits = list(repo.iter_commits(reverse=True))
total_commits = len(chronological_commits)

print(f"Discovered a total of {total_commits} historical commits inside the repo history.\n")
print("=" * 60)

ingested_count = 0
skipped_count = 0
LIMIT = -1

# 1. Slice target scope up to our execution LIMIT
target_commits = chronological_commits[:LIMIT]

# 2. Initialize the tqdm context manager manually targeted to our ingestion ceiling
with tqdm(total=LIMIT, desc="Pipeline Ingestion", unit="commit") as pbar:
    
    for idx, commit in enumerate(target_commits, start=1):
        commit_id = commit.hexsha
        
        # Extract rich repo analytics context (filters for .java files)
        raw_payload = data_loader.fetch_commit_data(project=PROJECT_KEY, commit_id=commit_id)
        
        # Skip if the commit didn't alter any Java source blocks
        if raw_payload is None:
            skipped_count += 1
            # Dynamically update metadata counter on the right side of the progress bar
            pbar.set_postfix(ingested=ingested_count, skipped_non_java=skipped_count)
            pbar.update(1)
            continue
            
        # Inject directly via our object-oriented rule map engine
        success = kg.ingest(raw_payload)
        
        if success:
            ingested_count += 1
            
            # Use pbar.set_description to flash the currently processing Commit ID over the bar
            pbar.set_description(f"Ingested {commit_id[:8]}")
            
            # Dynamically update progress counters in real-time
            pbar.set_postfix(ingested=ingested_count, skipped_non_java=skipped_count)
            pbar.update(1)
        else:
            # Handle case where transaction failed or dropped out
            pbar.write(f"⚠️  Database transaction failed on commit block: {commit_id[:10]}")
            pbar.update(1)

print("=" * 60)
print("Pipeline processing sequence finalized successfully!")

Wiping historical graph database state...

Opening local repository at: E:/repos/groovy
Extracting commit timeline in ascending chronological order (oldest -> newest)...
Discovered a total of 22601 historical commits inside the repo history.

Wiping historical graph database state...

Opening local repository at: E:/repos/groovy
Extracting commit timeline in ascending chronological order (oldest -> newest)...
Discovered a total of 22601 historical commits inside the repo history.



Ingested e3c8eb37: : 6313commit [1:19:02,  2.75commit/s, ingested=4193, skipped_non_java=2121] 2026-06-10 13:08:14,725 - ERROR - Failed to ingest transaction into Neo4j: Existing exports of data: object cannot be re-sized
Ingested e3c8eb37: : 6315commit [1:19:03,  2.09commit/s, ingested=4193, skipped_non_java=2121]

⚠️  Database transaction failed on commit block: dacee3d220


Ingested 341a0859: : 6369commit [1:19:29,  2.54commit/s, ingested=4239, skipped_non_java=2129]

In [24]:
# =====================================================================
# GRAPH METRICS SUMMARY VERIFICATION
# =====================================================================
print("=" * 60)
print("\n--- Processing Pipeline Summary ---")
print(f"Total Evaluated Repo Commits      : {total_commits}")
print(f"Successfully Vectorized (Java)    : {ingested_count}")
print(f"Skipped (Non-Java/Docs/Configs)   : {skipped_count}")

def check_graph_labels(tx):
    query = "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS total ORDER BY total DESC"
    return [row.data() for row in tx.run(query)]

with kg.driver.session() as session:
    distribution = session.execute_read(check_graph_labels)

print("\n--- Neo4j Local Database Topology Status ---")
if not distribution:
    print("Database is currently empty.")
for node_metric in distribution:
    print(f"Node Label: {node_metric['label']:15} | Node Count: {node_metric['total']}")

# Clean up memory resources and close thread channels safely
kg.close()
print("\nSession completed successfully.")


--- Processing Pipeline Summary ---
Total Evaluated Repo Commits      : 22601
Successfully Vectorized (Java)    : 56
Skipped (Non-Java/Docs/Configs)   : 44

--- Neo4j Local Database Topology Status ---
Node Label: File            | Node Count: 130
Node Label: Commit          | Node Count: 56
Node Label: Developer       | Node Count: 2
Node Label: Project         | Node Count: 1

Session completed successfully.
